In [ ]:
import pandas as pd
from pathlib import Path

import numpy as np
import rasterio
from rasterio.windows import from_bounds
from rasterio.enums import Resampling

In [ ]:
DIR_DATA = Path("data")
DIR_METADATA = DIR_DATA / "0_metadata"
DIR_SENTINEL2_ORG = DIR_DATA / "1_org" / "2_Sentinel2"
DIR_SENTINEL2_PROCESSED = DIR_DATA / "2_processed" / "2_Sentinel2"

FILEPATH_PATCH_CENTERS = DIR_METADATA / "patch-centers.csv"
FILEPATH_SENTINEL2_LIST = DIR_METADATA / "sentinel2-list.csv"

SAR_RESOLUTION = 0.5
PATCH_SIZE = 256
GROUND_SIZE = PATCH_SIZE * SAR_RESOLUTION  # 128 m
HALF_SIZE = GROUND_SIZE / 2  # 64 m

In [ ]:
flag_test = None  # 1: used for test data, otherwise not

patches = pd.read_csv(FILEPATH_PATCH_CENTERS)
# Remove empty rows
patches = patches.dropna(subset=["id"])

images = pd.read_csv(FILEPATH_SENTINEL2_LIST)

for _, img in images.iterrows():
    # Skip images that are never used
    if img["count"] == 0:
        continue

    image_name = img["name"]
    print(f"Processing {image_name} ...")

    image_path = DIR_SENTINEL2_ORG / f"{image_name}.tif"

    image_year = pd.to_datetime(img["date"]).year
    dir_target = DIR_SENTINEL2_PROCESSED / f"patches_{image_year}" / image_name
    dir_target.mkdir(parents=True, exist_ok=True)

    # Read test flag
    flag_test = img["test"]

    with rasterio.open(image_path) as src:
        assert src.crs.to_epsg() == 2958

        for _, patch in patches.iterrows():
            # Skip patches that don't belong to the current split
            if patch["usage"] == "test" and flag_test != 1:
                continue

            print(f"Patch: {patch['id']}, Year: {patch['year']}")
            if int(patch["year"]) != image_year:
                print(f"Patch {patch['id']} is not from {image_year}")
                continue

            x = patch["x"]
            y = patch["y"]

            xmin = x - HALF_SIZE
            xmax = x + HALF_SIZE
            ymin = y - HALF_SIZE
            ymax = y + HALF_SIZE

            window = from_bounds(
                xmin,
                ymin,
                xmax,
                ymax,
                transform=src.transform
            )
            window = window.round_offsets().round_lengths()

            if (
                window.col_off < 0 or
                window.row_off < 0 or
                window.col_off + window.width > src.width or
                window.row_off + window.height > src.height
            ):
                print(f"{patch['id']} is out of bounds.")
                continue

            patch_img = src.read(
                window=window,
                out_shape=(
                    src.count,
                    PATCH_SIZE,
                    PATCH_SIZE
                ),
                resampling=Resampling.bilinear
            )

            # Read only the Red and NIR bands
            red = patch_img[2].astype(np.float32)   # B4
            nir = patch_img[3].astype(np.float32)   # B8

            # Calculate NDVI
            ndvi = (nir - red) / (nir + red + 1e-6)

            profile = src.profile.copy()
            profile.update(
                count=1,
                dtype="float32",
                width=PATCH_SIZE,
                height=PATCH_SIZE,
                transform=rasterio.transform.from_bounds(
                    xmin,
                    ymin,
                    xmax,
                    ymax,
                    PATCH_SIZE,
                    PATCH_SIZE
                ),
                driver="GTiff",
                compress="LZW"
            )
            output = dir_target / f"{image_name}_{patch['id']}_NDVI.tif"

            with rasterio.open(output, "w", **profile) as dst:
                dst.write(ndvi, 1)
